# Robust Portfolio Engineering: Navigating the 2022 Correlation Crisis

This notebook implements and compares three portfolio optimization methods during the 2022 market crisis when traditional correlations broke down. We'll examine:

1. **Mean-Variance Optimization (MVO)** - Traditional approach
2. **Hierarchical Risk Parity (HRP)** - Robust to correlation breakdowns
3. **CVaR Optimization** - Focuses on tail risk

In [ ]:
# Import required libraries
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to path to import our modules
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

# Import our custom modules
from data import DataLoader
from engine import PortfolioEngine

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

## Data Loading and Preprocessing

First, we'll load 5 years of historical data for our key assets: SPY, TLT, GLD, and BTC-USD.

In [ ]:
# Define assets and time period
assets = ['SPY', 'TLT', 'GLD', 'BTC-USD']
start_date = '2019-01-01'
end_date = '2024-01-01'

# Initialize data loader
loader = DataLoader(symbols=assets, start_date=start_date, end_date=end_date)

# Get price and return data
data_dict = loader.get_data(force_download=False)
prices = data_dict['prices']
returns = data_dict['returns']

print(f"Data shape: {returns.shape}")
print(f"Date range: {returns.index[0]} to {returns.index[-1]}")
print(f"Assets: {list(returns.columns)}")

## The 2022 Correlation Breakdown

Let's analyze the correlation breakdown that occurred in 2022, particularly between SPY and TLT which traditionally had negative correlation.

In [ ]:
# Calculate rolling correlations between SPY and TLT
rolling_window = 60  # 60-day rolling window
spy_tlt_corr = returns[['SPY', 'TLT']].rolling(rolling_window).corr().unstack()['SPY']['TLT']

# Plot rolling correlation
plt.figure(figsize=(14, 7))
plt.plot(spy_tlt_corr.index, spy_tlt_corr.values, linewidth=2)
plt.title(f'60-Day Rolling Correlation: SPY vs TLT', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Correlation', fontsize=12)
plt.axhline(y=0, color='k', linestyle='--', alpha=0.5)
plt.axvline(x=pd.Timestamp('2022-01-01'), color='red', linestyle=':', alpha=0.7, label='2022 Start')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Highlight the 2022 crisis period
print("Average correlation (pre-2022):", spy_tlt_corr[:'2021-12-31'].mean())
print("Average correlation (2022):", spy_tlt_corr['2022-01-01':'2022-12-31'].mean())
print("Average correlation (post-2022):", spy_tlt_corr['2023-01-01':].mean())

## Portfolio Optimization Models

Now we'll implement and compare our three portfolio optimization models using the PortfolioEngine.

In [ ]:
# Initialize portfolio engine
engine = PortfolioEngine(returns)

# Calculate portfolio weights using each method
hrp_weights = engine.hrp_optimization()
cvar_weights = engine.cvar_optimization(alpha=0.05)
mvo_weights = engine.mvo_optimization()

# Display the weights
weights_df = pd.DataFrame({
    'HRP': hrp_weights,
    'CVaR': cvar_weights,
    'MVO': mvo_weights
})

print("Portfolio Weights Comparison:")
print(weights_df.round(4))

## Dendrogram Visualization (HRP Clustering)

Let's visualize the hierarchical clustering structure used in HRP.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Calculate distance matrix for hierarchical clustering
corr_matrix = engine.correlation_matrix
distance_matrix = np.sqrt(2 * (1 - corr_matrix))

# Perform hierarchical clustering
link_matrix = linkage(distance_matrix, method='single')

# Plot dendrogram
plt.figure(figsize=(12, 6))
dendrogram(link_matrix, labels=engine.assets, leaf_rotation=45)
plt.title('Hierarchical Clustering Dendrogram (HRP)', fontsize=16)
plt.xlabel('Assets', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.tight_layout()
plt.show()

## Out-of-Sample Performance Comparison

Let's compare the performance of each strategy using an out-of-sample period (2022-2023).

In [ ]:
# Define in-sample and out-of-sample periods
in_sample_end = '2021-12-31'
out_sample_start = '2022-01-01'

# Split returns
in_sample_returns = returns[:in_sample_end]
out_sample_returns = returns[out_sample_start:]

# Initialize engines for each period
in_sample_engine = PortfolioEngine(in_sample_returns)
out_sample_engine = PortfolioEngine(out_sample_returns)

# Calculate weights using in-sample data
hrp_weights_is = in_sample_engine.hrp_optimization()
cvar_weights_is = in_sample_engine.cvar_optimization(alpha=0.05)
mvo_weights_is = in_sample_engine.mvo_optimization()

# Calculate out-of-sample metrics
hrp_metrics = out_sample_engine.calculate_portfolio_metrics(hrp_weights_is, out_sample_returns)
cvar_metrics = out_sample_engine.calculate_portfolio_metrics(cvar_weights_is, out_sample_returns)
mvo_metrics = out_sample_engine.calculate_portfolio_metrics(mvo_weights_is, out_sample_returns)

# Create metrics comparison DataFrame
metrics_df = pd.DataFrame({
    'HRP': hrp_metrics,
    'CVaR': cvar_metrics,
    'MVO': mvo_metrics
})

print("Out-of-Sample Performance Metrics:")
print(metrics_df.round(4))

## Cumulative Returns Comparison

Visualizing the cumulative performance of each strategy.

In [ ]:
# Calculate out-of-sample portfolio returns for each strategy
hrp_returns_os = (out_sample_returns * pd.Series(hrp_weights_is)).sum(axis=1)
cvar_returns_os = (out_sample_returns * pd.Series(cvar_weights_is)).sum(axis=1)
mvo_returns_os = (out_sample_returns * pd.Series(mvo_weights_is)).sum(axis=1)

# Calculate cumulative returns
hrp_cum_returns = (1 + hrp_returns_os).cumprod()
cvar_cum_returns = (1 + cvar_returns_os).cumprod()
mvo_cum_returns = (1 + mvo_returns_os).cumprod()

# Plot cumulative returns
plt.figure(figsize=(14, 8))
plt.plot(hrp_cum_returns.index, hrp_cum_returns.values, label='HRP', linewidth=2)
plt.plot(cvar_cum_returns.index, cvar_cum_returns.values, label='CVaR', linewidth=2)
plt.plot(mvo_cum_returns.index, mvo_cum_returns.values, label='MVO', linewidth=2)
plt.title('Out-of-Sample Cumulative Returns Comparison', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Return', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Drawdown Analysis

Let's analyze the drawdowns for each strategy to understand risk characteristics.

In [ ]:
# Calculate drawdowns
def calculate_drawdown(returns):
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    return drawdown

hrp_drawdown = calculate_drawdown(hrp_returns_os)
cvar_drawdown = calculate_drawdown(cvar_returns_os)
mvo_drawdown = calculate_drawdown(mvo_returns_os)

# Plot drawdowns
plt.figure(figsize=(14, 8))
plt.fill_between(hrp_drawdown.index, hrp_drawdown.values, 0, alpha=0.3, label='HRP')
plt.fill_between(cvar_drawdown.index, cvar_drawdown.values, 0, alpha=0.3, label='CVaR')
plt.fill_between(mvo_drawdown.index, mvo_drawdown.values, 0, alpha=0.3, label='MVO')
plt.title('Out-of-Sample Drawdown Comparison', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Drawdown', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print max drawdowns
print(f"Max Drawdowns:")
print(f"HRP: {hrp_drawdown.min():.4f}")
print(f"CVaR: {cvar_drawdown.min():.4f}")
print(f"MVO: {mvo_drawdown.min():.4f}")

## Summary

This analysis demonstrates how different portfolio optimization methods perform during periods of market stress like the 2022 correlation crisis. The Hierarchical Risk Parity (HRP) approach typically shows more stability during such periods by taking into account the hierarchical structure of asset correlations, while traditional MVO can be sensitive to correlation breakdowns. The CVaR approach focuses on tail risk, which can be beneficial during crisis periods.